# Train balance bot using PPO with curriculum learning

Run each cell by pressing `shift + enter`.

In this notebook, we configure PPO and the balance bot environment (gymnasium wrapper for MuJoCo). We perform a simple 2-step curriculum learning. In the first phase, we train the bot to stay upright in any way possible. This will usually result in the bot driving around to stay up. In the second phase, we introduce a penalty for moving off the original location or rotating about the Z axis (yaw).

Note that we're using *privileged information* in the second phase. We rely on MuJoCo's `qpos` and `qvel` variables to get the robot's position and velocity information without using any built-in sensors. This helps shape the reward function, but it's not something we have easy access to on the real robot.

In [1]:
# Import standard libraries
from dataclasses import replace
import os
from pathlib import Path
import sys
import time

# Third-party libraries
import gymnasium as gym
from gymnasium.wrappers import RecordEpisodeStatistics
import numpy as np
import torch

# Import custom environment (NU54 copy in this folder)
sys.path.insert(0, "/workspace/nu54/02-balance-bot-with-ppo")
from balance_bot_env import BalanceBotEnv

# Add the folder containing our envs/ and rl/ packages to the path
sys.path.append("/workspace/software")

# Import PPO training module and exporter
from rl.ppo_trainer import PPOConfig, evaluate, train, export_tb_plots_as_csv, export_actor_onnx

# NU54: train() writes to runs/ relative to the working directory, and TensorBoard
# in the container only watches /workspace/software. Work from there so the runs
# show up in TensorBoard (runs/ is gitignored, so no tracked files change).
os.chdir("/workspace/software")

In [2]:
# Settings
MJCF_PATH = Path("/workspace/nu54/FreeCAD/nu54/nu54.xml")
SEED = 42
NUM_ENVS = 4               # Number of parallel environments. Only the first will be rendered.
STEPS_PER_ENV = 500_000    # Number of simulation steps to perform per environment

In [3]:
# Configure PPO
ppo_config = PPOConfig(
    exp_name = "nu54-balance-ppo", # Name of the experiment (NU54 model)
    env_id = "BalanceBot-v0",      # Name of the environment
    seed = SEED,                   # Constant seed for reproducibility
    num_envs = NUM_ENVS,           # Number of parallel environments
    actor_hidden_layers = 2,       # Number of hidden layers in the actor network
    actor_hidden_size = 16,        # Number of nodes in each hidden layer in the actor
    critic_hidden_layers = 2,      # Number of hidden layers in the critic network
    critic_hidden_size = 16,       # Number of nodes in each hidden layer in the critic
    total_timesteps = NUM_ENVS * STEPS_PER_ENV,  # Total simulation steps (all envs and iterations)
    num_steps = 2048,              # Number of steps per rollout per env (2048 * 0.002s = ~4 sec)
    num_minibatches = 32,          # Number of minibatches for each training epoch
    update_epochs = 10,            # Number of epochs to update actor and critic for each iteration
    anneal_lr = True,              # Enable annealing (lower learning rate as training goes on)
    learning_rate = 3e-4,          # Initial learning rate, reduced by annealing (if enabled)
    gamma = 0.99,                  # Discount factor (future rewards are discounted by this amount)
    gae_lambda = 0.95,             # GAE blending: 0 = pure TD error, 1 = pure Monte Carlo
    clip_coef = 0.2,               # Limits policy ratio to prevent large actor updates
    value_clip = 1.0,              # Absolute bounds on value prediction change per update (critic)
    ent_coef = 0.0,                # How much entropy factors into total loss calculation
    vf_coef = 0.5,                 # How much the value loss factors into total loss calculation
    max_grad_norm = 0.5,           # Limits how much actor/critic parameters can change during an update
    checkpoint_interval = 50,      # Save model every 50 iterations
    save_model = True,             # Save the final model
    timestep = 0.000,              # Match MJCF opt.timestep for real-time rendering (or 0 for fast)
)

In [4]:
def make_balance_bot_env(render, **kwargs):
    """Function to create an environment for our balance bot"""
    # Create the environment and set the render mode
    env = BalanceBotEnv(
        mjcf_path    = MJCF_PATH,
        render_mode  = "human" if render else None,
        **kwargs
    )

    # Wrap in RecordEpisodeStatistics so we can log episodic returns in the 'info' dict
    return gym.wrappers.RecordEpisodeStatistics(env)

def make_envs(num_envs, **kwargs):
    """Create a SyncVectorEnv with num_envs balance bot environments."""
    env_factories = []
    for i in range(num_envs):
        env_factories.append(
            lambda render=(i==0), kw=kwargs: make_balance_bot_env(render, **kw)
        )
    return gym.vector.SyncVectorEnv(env_factories)

In [8]:
# Phase 1: Balance only (don't worry about position or rotation)
envs = make_envs(
    NUM_ENVS,
    pitch_penalty_coef=0.5,
    action_penalty_coef=0.01,
    position_penalty_coef=0.0,
    yaw_penalty_coef=0.0
)

# Choo choo train
result = train(ppo_config, envs=envs)

Run name: BalanceBot-v0__nu54-balance-ppo__42__1795413215
TensorBoard: http://localhost:6006/#scalars&regexFilter=nu54-balance-ppo
Checkpoint saved to runs/BalanceBot-v0__nu54-balance-ppo__42__1795413215/checkpoint_iter0050.cleanrl_model
New best model saved (mean_return=1166.93) to runs/BalanceBot-v0__nu54-balance-ppo__42__1795413215/best_model.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__nu54-balance-ppo__42__1795413215/checkpoint_iter0100.cleanrl_model
New best model saved (mean_return=8344.29) to runs/BalanceBot-v0__nu54-balance-ppo__42__1795413215/best_model.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__nu54-balance-ppo__42__1795413215/checkpoint_iter0150.cleanrl_model
New best model saved (mean_return=9961.84) to runs/BalanceBot-v0__nu54-balance-ppo__42__1795413215/best_model.cleanrl_model


NaN or Inf found in input tensor.


Checkpoint saved to runs/BalanceBot-v0__nu54-balance-ppo__42__1795413215/checkpoint_iter0200.cleanrl_model
New best model saved (mean_return=9986.13) to runs/BalanceBot-v0__nu54-balance-ppo__42__1795413215/best_model.cleanrl_model
Final model saved to runs/BalanceBot-v0__nu54-balance-ppo__42__1795413215/nu54-balance-ppo_final.cleanrl_model
Best model mean return: 9986.13, saved to runs/BalanceBot-v0__nu54-balance-ppo__42__1795413215/best_model.cleanrl_model


In [9]:
# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Best model: runs/BalanceBot-v0__nu54-balance-ppo__42__1795413215/best_model.cleanrl_model
Final model: runs/BalanceBot-v0__nu54-balance-ppo__42__1795413215/nu54-balance-ppo_final.cleanrl_model
Best mean return: 9986.13
Loaded best model (mean_return=9986.13)


In [10]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

Mean return: 2043.96


In [11]:
# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

Exported charts_metrics.csv (5 metrics, 2337 steps)
Exported losses_metrics.csv (7 metrics, 244 steps)


In [12]:
# Phase 2: Update the position and rotation coefficients in the existing environments
for env_stat_wrapper in envs.envs:
    env = env_stat_wrapper.env
    env.position_penalty_coef = 0.001
    env.yaw_penalty_coef = 0.1

# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

Run name: BalanceBot-v0__nu54-balance-ppo__42__1795414415
TensorBoard: http://localhost:6006/#scalars&regexFilter=nu54-balance-ppo
Checkpoint saved to runs/BalanceBot-v0__nu54-balance-ppo__42__1795414415/checkpoint_iter0050.cleanrl_model
New best model saved (mean_return=9964.39) to runs/BalanceBot-v0__nu54-balance-ppo__42__1795414415/best_model.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__nu54-balance-ppo__42__1795414415/checkpoint_iter0100.cleanrl_model
New best model saved (mean_return=9971.00) to runs/BalanceBot-v0__nu54-balance-ppo__42__1795414415/best_model.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__nu54-balance-ppo__42__1795414415/checkpoint_iter0150.cleanrl_model
New best model saved (mean_return=9974.11) to runs/BalanceBot-v0__nu54-balance-ppo__42__1795414415/best_model.cleanrl_model
Checkpoint saved to runs/BalanceBot-v0__nu54-balance-ppo__42__1795414415/checkpoint_iter0200.cleanrl_model
New best model saved (mean_return=9987.48) to runs/BalanceBot-v0__nu5

In [13]:
# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Best model: runs/BalanceBot-v0__nu54-balance-ppo__42__1795414415/best_model.cleanrl_model
Final model: runs/BalanceBot-v0__nu54-balance-ppo__42__1795414415/nu54-balance-ppo_final.cleanrl_model
Best mean return: 9987.48
Loaded best model (mean_return=9987.48)


In [14]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

Mean return: 2044.37


In [15]:
# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

Exported charts_metrics.csv (5 metrics, 293 steps)
Exported losses_metrics.csv (7 metrics, 244 steps)


In [16]:
# Close the environments
for idx, env in enumerate(envs.envs):
    print(f"Closing env {idx}")
    env.env.close()

Closing env 0
Closing env 1
Closing env 2
Closing env 3


In [17]:
# Get observation and action sizes
obs_size = envs.single_observation_space.shape[0]
action_size = envs.single_action_space.shape[0]

# Export the actor network as an ONNX model
export_actor_onnx(
    model_path=result.best_model_path,
    output_path=result.checkpoint_dir / "actor.onnx",
    obs_size=obs_size,
    action_size=action_size,
    num_hidden_layers=ppo_config.actor_hidden_layers,
    hidden_layer_size=ppo_config.actor_hidden_size,
)

/workspace/software/rl/ppo_trainer.py:381: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W1123 06:31:57.216000 60904 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::nms
W1123 06:31:57.217000 60904 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::roi_align
W1123 06:31:57.218000 60904 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::roi_pool


[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Actor exported to ONNX: runs/BalanceBot-v0__nu54-balance-ppo__42__1795414415/actor.onnx


/opt/pyenv/versions/3.12.16/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


## NU54: 50-second evaluation of both phases

The original `evaluate()` reads the episode limit with `getattr(envs.envs[0], 'max_steps', config.num_steps)`.
`envs[0]` is the `RecordEpisodeStatistics` wrapper, which (gymnasium 1.2) does not forward `max_steps`,
so the evaluation above stopped at `num_steps` = 2048 steps (10.24 s). Here we pass `num_steps=10_000`
(50 s at the 5 ms physics timestep in `nu54.xml`). `num_steps` is only a fallback in `evaluate()`,
so training is not affected.

The environments were closed above, so we build a fresh single (rendered) environment for each phase
and load that phase's best model from disk. Each phase uses its own reward coefficients so the
returns are comparable with training. Note: `evaluate()` samples actions from the Gaussian policy;
the exported ONNX actor (and the real robot) uses the mean action.

In [18]:
# NU54: run folders from this session (Phase 1, Phase 2)
EVAL_RUNS = {
    "Phase 1": (Path("runs/BalanceBot-v0__nu54-balance-ppo__42__1795413215"),
                dict(position_penalty_coef=0.0, yaw_penalty_coef=0.0)),
    "Phase 2": (Path("runs/BalanceBot-v0__nu54-balance-ppo__42__1795414415"),
                dict(position_penalty_coef=0.001, yaw_penalty_coef=0.1)),
}
EVAL_EPISODES = 3
EVAL_MAX_STEPS = 10_000    # 10000 steps * 0.005 s = 50 s

from rl.ppo_trainer import Agent

def load_phase(run_dir, coefs, render):
    """Build one environment with the phase's reward coefficients and load its best model"""
    eval_envs = make_envs(1, pitch_penalty_coef=0.5, action_penalty_coef=0.01, **coefs) if render else \
        gym.vector.SyncVectorEnv([lambda: make_balance_bot_env(False, pitch_penalty_coef=0.5,
                                                               action_penalty_coef=0.01, **coefs)])
    agent = Agent(eval_envs, ppo_config)
    agent.load_state_dict(torch.load(run_dir / "best_model.cleanrl_model", weights_only=True))
    return agent, eval_envs

eval_config_50s = replace(ppo_config, timestep=0.005, num_steps=EVAL_MAX_STEPS)

eval_results = {}
for name, (run_dir, coefs) in EVAL_RUNS.items():
    print(f"{name}: evaluating {EVAL_EPISODES} episodes (up to {EVAL_MAX_STEPS * 0.005:.0f} s each, real time)...")
    agent, eval_envs = load_phase(run_dir, coefs, render=True)
    eval_results[name] = evaluate(agent, eval_episodes=EVAL_EPISODES, config=eval_config_50s, envs=eval_envs)
    eval_envs.envs[0].env.close()
    print(f"{name}: returns {np.round(eval_results[name], 1)}, mean {np.nanmean(eval_results[name]):.2f}")

Phase 1: evaluating 3 episodes (up to 50 s each, real time)...
Phase 1: returns [9980.6 9979.5 9980.9], mean 9980.36
Phase 2: evaluating 3 episodes (up to 50 s each, real time)...
Phase 2: returns [9974.4 9997.  9998. ], mean 9989.82


### NU54: how big are the actions? (prediction 2)

Runs one 50 s episode per phase without rendering (fast) and uses the **mean** action, which is what
`actor.onnx` outputs on the robot. The model's static friction keeps the wheels from turning below
about ctrl 0.12 (`frictionloss` in `nu54.xml`). Only numbers are printed; interpreting them is up to you.

In [19]:
DEADZONE = 0.12

for name, (run_dir, coefs) in EVAL_RUNS.items():
    agent, eval_envs = load_phase(run_dir, coefs, render=False)
    obs, _ = eval_envs.reset(seed=SEED)
    actions, pitches = [], []
    for step in range(EVAL_MAX_STEPS):
        with torch.no_grad():
            action = agent.actor_mean(torch.Tensor(obs)).numpy()
        actions.append(action[0])
        pitches.append(obs[0][0])
        obs, _, terminated, truncated, _ = eval_envs.step(action)
        if terminated[0] or truncated[0]:
            break
    eval_envs.close()

    a = np.abs(np.clip(np.array(actions), -1.0, 1.0))   # MuJoCo clamps ctrl to [-1, 1]
    print(f"{name}: {len(actions)} steps ({len(actions) * 0.005:.1f} s)")
    print(f"  |ctrl| median {np.median(a):.3f}, 90th pct {np.percentile(a, 90):.3f}, max {a.max():.3f}")
    print(f"  share of steps with |ctrl| < {DEADZONE}: {np.mean(a < DEADZONE) * 100:.1f} %, "
          f"at the limit (|ctrl| >= 0.99): {np.mean(a >= 0.99) * 100:.1f} %")
    print(f"  |pitch| median {np.degrees(np.median(np.abs(pitches))):.2f} deg, "
          f"max {np.degrees(np.max(np.abs(pitches))):.2f} deg")

Phase 1: 10000 steps (50.0 s)
  |ctrl| median 0.200, 90th pct 0.291, max 0.444
  share of steps with |ctrl| < 0.12: 3.1 %, at the limit (|ctrl| >= 0.99): 0.0 %
  |pitch| median 0.85 deg, max 2.67 deg
Phase 2: 10000 steps (50.0 s)
  |ctrl| median 0.010, 90th pct 0.021, max 0.351
  share of steps with |ctrl| < 0.12: 97.4 %, at the limit (|ctrl| >= 0.99): 0.0 %
  |pitch| median 0.62 deg, max 2.64 deg


### NU54: how far does it drive, and how big is the circle?

Same 50 s episode (no rendering), run twice per phase: with the **mean** action (what `actor.onnx` outputs on the robot) and with a **sampled** action (what `evaluate()` does, i.e. the rendered video). Uses privileged simulator state that the policy
never sees: the free joint position `qpos[0:2]` (x, y in m), the body orientation quaternion `qpos[3:7]`
(yaw), and `qvel[5]` (yaw rate, the same value the Phase 2 yaw penalty uses). The circle is a least-squares
fit to the x-y path after the first 10 s. Only numbers are printed.

In [ ]:
def yaw_from_quat(q):
    """Yaw (rotation about world Z) from a MuJoCo quaternion [w, x, y, z]"""
    w, x, y, z = q
    return np.arctan2(2 * (w * z + x * y), 1 - 2 * (y * y + z * z))

def fit_circle(x, y):
    """Least-squares circle fit, returns (cx, cy, r)"""
    A = np.column_stack([x, y, np.ones_like(x)])
    b = x**2 + y**2
    (a0, a1, a2), *_ = np.linalg.lstsq(A, b, rcond=None)
    cx, cy = a0 / 2, a1 / 2
    return cx, cy, np.sqrt(a2 + cx**2 + cy**2)

DT = 0.005
SKIP = int(10 / DT)    # ignore the first 10 s when fitting the circle

for name, (run_dir, coefs), mode in [(n, rc, m) for n, rc in EVAL_RUNS.items() for m in ("mean", "sampled")]:
    agent, eval_envs = load_phase(run_dir, coefs, render=False)
    torch.manual_seed(SEED)
    sim = eval_envs.envs[0].unwrapped
    obs, _ = eval_envs.reset(seed=SEED)
    xy, yaw, yaw_rate, wheel_vel, ctrl = [], [], [], [], []
    for step in range(EVAL_MAX_STEPS):
        with torch.no_grad():
            if mode == "mean":     # what actor.onnx outputs on the robot
                action = agent.actor_mean(torch.Tensor(obs)).numpy()
            else:                  # what evaluate() (the rendered video) does
                action = agent.get_action_and_value(torch.Tensor(obs))[0].numpy()
        obs, _, terminated, truncated, _ = eval_envs.step(action)
        if terminated[0] or truncated[0]:
            break
        xy.append(sim.data.qpos[0:2].copy())
        yaw.append(yaw_from_quat(sim.data.qpos[3:7]))
        yaw_rate.append(sim.data.qvel[5])
        wheel_vel.append(obs[0][2:4].copy())
        ctrl.append(np.clip(action[0], -1, 1))
    eval_envs.close()

    xy, wheel_vel, ctrl = np.array(xy), np.array(wheel_vel), np.array(ctrl)
    yaw = np.unwrap(np.array(yaw))
    path = np.sum(np.linalg.norm(np.diff(xy, axis=0), axis=1))
    dist = np.linalg.norm(xy, axis=1)
    cx, cy, r = fit_circle(xy[SKIP:, 0], xy[SKIP:, 1])
    resid = np.abs(np.hypot(xy[SKIP:, 0] - cx, xy[SKIP:, 1] - cy) - r)

    print(f"{name}, {mode} action: {len(xy) * DT:.1f} s")
    print(f"  path length {path:.2f} m (mean speed {path / (len(xy) * DT):.3f} m/s), "
          f"distance from start: final {dist[-1]:.3f} m, max {dist.max():.3f} m")
    print(f"  yaw: total {np.degrees(yaw[-1] - yaw[0]):.0f} deg ({(yaw[-1] - yaw[0]) / (2 * np.pi):.2f} turns), "
          f"yaw rate mean {np.mean(yaw_rate):.3f} rad/s, mean |yaw rate| {np.mean(np.abs(yaw_rate)):.3f} rad/s")
    if np.ptp(xy[SKIP:], axis=0).max() > 0.05:   # a circle fit needs the robot to actually move
        print(f"  circle fit (after 10 s): radius {r:.3f} m, center ({cx:.3f}, {cy:.3f}) m, "
              f"mean fit error {resid.mean() * 1000:.1f} mm")
    else:
        print("  circle fit: skipped (moved less than 5 cm after 10 s)")
    print(f"  wheel speed mean L {wheel_vel[:, 0].mean():.3f} / R {wheel_vel[:, 1].mean():.3f} rad/s, "
          f"ctrl mean L {ctrl[:, 0].mean():.3f} / R {ctrl[:, 1].mean():.3f}")